In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pycountry

In [3]:
# Paths
EUROSTAT_FILE = "../Raw/nasa_10_f_bs__custom_21212140_spreadsheet.xlsx"
CPIS_FILE = "../Clean/IMF_CPIS.csv"

# Eurostat is downloaded in million EUR.
# CPIS in your file appears to be raw USD, so we convert it to million USD below.
CPIS_IS_RAW_USD = True

In [4]:
EU27 = [
    "AUT", "BEL", "BGR", "CZE", "DNK", "EST", "FIN", "FRA", "DEU", "GRC",
    "HUN", "ITA", "LVA", "LTU", "LUX", "MLT", "NLD", "POL", "PRT", "ROU",
    "SVK", "SVN", "ESP", "SWE", "HRV", "CYP", "IRL"
]

# Add the UK, NOR, and CHE for completeness, even though they are not in the EU.
EU27 += ["GBR", "NOR", "CHE"]
# And USA
EU27 += ["USA"]

VALID_ISO3 = {c.alpha_3 for c in pycountry.countries}

# Eurostat country labels are not always identical to pycountry names, so use an explicit map.
EUROSTAT_COUNTRY_TO_ISO3 = {
    "Austria": "AUT",
    "Belgium": "BEL",
    "Bulgaria": "BGR",
    "Croatia": "HRV",
    "Cyprus": "CYP",
    "Czechia": "CZE",
    "Czech Republic": "CZE",
    "Denmark": "DNK",
    "Estonia": "EST",
    "Finland": "FIN",
    "France": "FRA",
    "Germany": "DEU",
    "Greece": "GRC",
    "Hungary": "HUN",
    "Ireland": "IRL",
    "Italy": "ITA",
    "Latvia": "LVA",
    "Lithuania": "LTU",
    "Luxembourg": "LUX",
    "Malta": "MLT",
    "Netherlands": "NLD",
    "Poland": "POL",
    "Portugal": "PRT",
    "Romania": "ROU",
    "Slovakia": "SVK",
    "Slovenia": "SVN",
    "Spain": "ESP",
    "Sweden": "SWE",
    "United Kingdom": "GBR",
    "Norway": "NOR",
    "Switzerland": "CHE",
    "United States": "USA"
}

In [5]:
# Load Eurostat workbook
xls = pd.ExcelFile(EUROSTAT_FILE)
print(xls.sheet_names)

['Summary', 'Sheet 1', 'Sheet 2', 'Sheet 3', 'Sheet 4', 'Sheet 5', 'Sheet 6', 'Sheet 7', 'Sheet 8', 'Sheet 9', 'Sheet 10']


/Users/jesper/Desktop/CBS/Thesis 1/Jesper-Liedholm-Thesis-Code/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
def clean_eurostat_sheet(sheet_name, value_name):
    df = pd.read_excel(xls, sheet_name, skiprows=11)

    # Drop empty/unnamed columns
    df = df.loc[:, ~df.columns.astype(str).str.contains("Unnamed", na=False)]
    df.columns = [str(c).strip() for c in df.columns]

    # First row is usually "GEO (Labels)" metadata
    df = df.drop(0).reset_index(drop=True)

    # Country column is called TIME in this Eurostat spreadsheet layout
    df = df.rename(columns={"TIME": "Country"})
    df["Country"] = df["Country"].astype(str).str.strip()

    # Remove EU aggregate rows. We want country rows only.
    df = df[~df["Country"].str.contains("European Union|Euro area", case=False, na=False)].copy()

    # Keep year columns
    year_cols = []
    for c in df.columns:
        c_str = str(c).strip()
        if c_str.isdigit() and 1990 <= int(c_str) <= 2035:
            year_cols.append(c)

    # Convert values to numeric
    for c in year_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Wide -> long
    out = df.melt(
        id_vars="Country",
        value_vars=year_cols,
        var_name="Year",
        value_name=value_name,
    )
    out["Year"] = out["Year"].astype(int)
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    return out

# Sheet 3 = Debt securities
# Sheet 4 = Listed shares
# Sheet 5 = Investment fund shares/units
sh3 = clean_eurostat_sheet("Sheet 3", "debt_securities_million_eur")
sh4 = clean_eurostat_sheet("Sheet 4", "listed_shares_million_eur")
sh5 = clean_eurostat_sheet("Sheet 5", "fund_shares_million_eur")

display(sh3.head(3))
display(sh4.head(3))
display(sh5.head(3))

,Country,Year,debt_securities_million_eur
0,Belgium,2016,652205.4
1,Bulgaria,2016,27638.3
2,Czechia,2016,126022.0


,Country,Year,listed_shares_million_eur
0,Belgium,2016,229796.8
1,Bulgaria,2016,4999.1
2,Czechia,2016,18291.4


,Country,Year,fund_shares_million_eur
0,Belgium,2016,328150.5
1,Bulgaria,2016,1504.7
2,Czechia,2016,22850.9


In [7]:
# Build Eurostat total resident portfolio holdings.
# These are already in million EUR.
# Sh3: Debt securities
# Sh4: Listed shares
# Sh5: Investment fund shares/units
df_euro = (
    sh3.merge(sh4, on=["Country", "Year"], how="outer")
       .merge(sh5, on=["Country", "Year"], how="outer")
)

for c in ["debt_securities_million_eur", "listed_shares_million_eur", "fund_shares_million_eur"]:
    df_euro[c] = pd.to_numeric(df_euro[c], errors="coerce")

# Keep missing as 0 only for summing. If all three are missing, total remains NaN.
component_cols = ["debt_securities_million_eur", "listed_shares_million_eur", "fund_shares_million_eur"]
all_missing = df_euro[component_cols].isna().all(axis=1)

df_euro["equity_fund_million_eur"] = (
    # df_euro["listed_shares_million_eur"].fillna(0)
    + df_euro["fund_shares_million_eur"].fillna(0)
)

df_euro["total_portfolio_million_eur"] = (
    df_euro["debt_securities_million_eur"].fillna(0)
    + df_euro["listed_shares_million_eur"].fillna(0)
    + df_euro["fund_shares_million_eur"].fillna(0)
)

df_euro.loc[all_missing, ["equity_fund_million_eur", "total_portfolio_million_eur"]] = np.nan

df_euro["iso3"] = df_euro["Country"].map(EUROSTAT_COUNTRY_TO_ISO3)
df_euro = df_euro[df_euro["iso3"].isin(EU27)].copy()

df_euro = df_euro.sort_values(["iso3", "Year"]).reset_index(drop=True)

display(df_euro.head(10))
print(df_euro.shape)

,Country,Year,debt_securities_million_eur,listed_shares_million_eur,fund_shares_million_eur,equity_fund_million_eur,total_portfolio_million_eur,iso3
0,Austria,2016,386474.3,101167.5,218520.0,218520.0,706161.8,AUT
1,Austria,2017,378410.8,125453.0,236884.3,236884.3,740748.1,AUT
2,Austria,2018,370892.2,108856.3,224415.5,224415.5,704164.0,AUT
3,Austria,2019,372814.4,130770.7,260347.2,260347.2,763932.3,AUT
4,Austria,2020,409656.3,139636.6,268499.8,268499.8,817792.7,AUT
5,Austria,2021,418703.1,187424.9,308702.3,308702.3,914830.3,AUT
6,Austria,2022,372916.5,166072.6,273340.9,273340.9,812330.0,AUT
7,Austria,2023,403810.3,183286.1,295323.0,295323.0,882419.4,AUT
8,Austria,2024,415754.0,189629.8,325605.2,325605.2,930989.0,AUT
9,Austria,2025,NaN,NaN,NaN,NaN,NaN,AUT


(300, 8)


In [8]:
# Get year-end USD/EUR exchange rates from ECB.
# ECB rate USD means: USD per 1 EUR.
url = "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-hist.zip"
fx = pd.read_csv(url)

fx["Date"] = pd.to_datetime(fx["Date"])
fx = fx.sort_values("Date")

year_end_fx = (
    fx.dropna(subset=["USD"])
      .assign(year=lambda x: x["Date"].dt.year)
      .groupby("year", as_index=False)
      .tail(1)
      [["year", "Date", "USD"]]
      .rename(columns={"Date": "fx_date", "USD": "usd_per_eur"})
      .reset_index(drop=True)
)

year_end_fx["eur_per_usd"] = 1 / year_end_fx["usd_per_eur"]

display(year_end_fx.tail(10))

,year,fx_date,usd_per_eur,eur_per_usd
18,2017,2017-12-29,1.1993,0.833820
19,2018,2018-12-31,1.1450,0.873362
20,2019,2019-12-31,1.1234,0.890155
21,2020,2020-12-31,1.2271,0.814930
22,2021,2021-12-31,1.1326,0.882924
23,2022,2022-12-30,1.0666,0.937559
24,2023,2023-12-29,1.1050,0.904977
25,2024,2024-12-31,1.0389,0.962557
26,2025,2025-12-31,1.1750,0.851064
27,2026,2026-04-28,1.1680,0.856164


In [9]:
# Convert Eurostat from million EUR to million USD.
# This is the correct direction: million USD = million EUR * USD per EUR.
df_euro = df_euro.merge(
    year_end_fx[["year", "usd_per_eur"]],
    left_on="Year",
    right_on="year",
    how="left",
).drop(columns="year")

for c in [
    "debt_securities_million_eur",
    "listed_shares_million_eur",
    "fund_shares_million_eur",
    "equity_fund_million_eur",
    "total_portfolio_million_eur",
]:
    df_euro[c.replace("_million_eur", "_million_usd")] = df_euro[c] * df_euro["usd_per_eur"]

display(df_euro[["iso3", "Country", "Year", "total_portfolio_million_eur", "usd_per_eur", "total_portfolio_million_usd"]].head(10))

,iso3,Country,Year,total_portfolio_million_eur,usd_per_eur,total_portfolio_million_usd
0,AUT,Austria,2016,706161.8,1.0541,7.443652e+05
1,AUT,Austria,2017,740748.1,1.1993,8.883792e+05
2,AUT,Austria,2018,704164.0,1.1450,8.062678e+05
3,AUT,Austria,2019,763932.3,1.1234,8.582015e+05
4,AUT,Austria,2020,817792.7,1.2271,1.003513e+06
5,AUT,Austria,2021,914830.3,1.1326,1.036137e+06
6,AUT,Austria,2022,812330.0,1.0666,8.664312e+05
7,AUT,Austria,2023,882419.4,1.1050,9.750734e+05
8,AUT,Austria,2024,930989.0,1.0389,9.672045e+05
9,AUT,Austria,2025,NaN,1.1750,NaN


In [30]:
# Save to csv
df_euro.to_csv("../Clean/Eurostat_total_portfolio.csv", index=False)